# GPU ablations on fold 0 (same code as the final pipeline)

ConvNeXt-tiny U-Nets are trained on folds 1–3 and evaluated on fold 0, identical except for one setting:

| variant | change | question |
|---|---|---|
| `ref` | final configuration | reference |
| `union` | target = agreement-weighted union of the annotator masks instead of their mean | does a recall-oriented target help? |
| `res1024` | whole disk downscaled to 1024 px instead of 1024 crops of the native 2048 px image | value of native resolution |
| `no_cldice` | clDice term removed from the loss | contribution of the topology loss |
| `ft20` | `ref` fine-tuned for 20 more epochs (new one-cycle schedule, max LR 1e-4) | does longer training help? |

All three run on the same hardware and precision, so the paired differences are not confounded by platform.
Evaluation: fold-0 images at full resolution with 4 flips → components → scorer cross-fitted over 4 image groups
inside fold 0 → PQ; paired bootstrap over images against `ref`. Decision rule fixed in advance: a change is
adopted only if its 95% CI is above zero and the gain exceeds +0.003.

The run is resumable: with a time budget the notebook stops cleanly; add this notebook's output as an input of the
next version and run again — finished models and interrupted states are picked up automatically.

In [ ]:
import os, sys, json, time, glob, pickle
from pathlib import Path
T_START = time.time()

REPO_DIR = Path(globals().get("REPO_DIR", ".."))
DATA_DIR = Path(globals().get("DATA_DIR", "/kaggle/input"))
WORK_DIR = Path(globals().get("WORK_DIR", "/kaggle/working/ablation"))
RESUME_DIRS = globals().get("RESUME_DIRS", ["/kaggle/input"])     # earlier outputs (finished models, states, cache)
TIME_BUDGET_H = globals().get("TIME_BUDGET_H", 11.3)              # Kaggle sessions end at 12 h
WORKERS = int(globals().get("WORKERS", max(1, min(4, os.cpu_count() or 2))))
DRY = os.environ.get("DRY") == "1"
FOLD = 0
VARIANTS = {"ref": {}, "union": {"data.target": "union"}, "res1024": {"data.input_res": 1024},
            "no_cldice": {"loss.w_cldice": 0.0},
            "ft20": {"train.epochs": 20, "train.lr": 1.0e-4, "loss.cldice_ramp": [-1, 0],   # clDice on from the start
                     "train.init_checkpoint": str(WORK_DIR / "ckpt" / f"abl_ref_fold{FOLD}.pt")}}
RUN = globals().get("RUN", list(VARIANTS))                        # e.g. ["union"] to split variants across sessions

sys.path.insert(0, str(REPO_DIR))
import numpy as np, pandas as pd, torch, sklearn
import matplotlib.pyplot as plt
from filament.config import load_config, scaled
from filament.data import Annotations, make_folds, build_cache, find_files
from filament.train import train_model
from filament import pipeline as pl, evaluate as ev

def variant_cfg(name):
    c = load_config(REPO_DIR / "configs" / "final.yaml", VARIANTS[name])
    if DRY:
        c = scaled(c, 256)
        c["train"].update(epochs=2, val_every=1, workers=0)
    return c

WORK_DIR.mkdir(parents=True, exist_ok=True)
log = pl.log_to(WORK_DIR / "ablation.log")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = variant_cfg("ref")
SEED = cfg["seed"]
log(f"device {device} ({torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'}) | torch {torch.__version__} "
    f"| sklearn {sklearn.__version__} | variants {RUN}")

In [ ]:
# --------------------------------------------------------------------------------------- data, folds, cache
ann = Annotations(find_files(DATA_DIR, "*kaggle2026_train.json")[0],
                  [d for d in find_files(DATA_DIR, "train_images") if os.path.isdir(d)][0])
folds_path = REPO_DIR / "splits" / "folds.csv"
folds = pd.read_csv(folds_path) if folds_path.exists() else make_folds(ann.meta, cfg["data"]["n_folds"], SEED)
folds = folds[folds.stem.isin(ann.meta.stem)].reset_index(drop=True)
TR = folds.stem[folds.fold != FOLD].tolist()
VA = folds.stem[folds.fold == FOLD].tolist()
STEMS = folds.stem.tolist()

def ready_cache(root):
    return all(len(glob.glob(f"{root}/{k}/*.png")) >= len(STEMS) for k in ("x", "y", "s"))
found = [Path(p).parent for p in glob.glob(f"{RESUME_DIRS[0]}/**/cache/x", recursive=True)] if RESUME_DIRS else []
found = [p for p in found if ready_cache(p)]
CACHE_DIR = found[0] if found and not DRY else WORK_DIR / "cache"
if CACHE_DIR == WORK_DIR / "cache":
    build_cache(ann, STEMS, CACHE_DIR, cfg, workers=WORKERS, log=log)
log(f"cache: {CACHE_DIR} | fold {FOLD}: {len(TR)} train / {len(VA)} validation images")

In [ ]:
# ------------------------------------------------------------------------------------------------- training
deadline = T_START + TIME_BUDGET_H * 3600 if TIME_BUDGET_H else None
STATUS = {}
for name in RUN:
    r = train_model(variant_cfg(name), f"abl_{name}_fold{FOLD}", TR, VA, CACHE_DIR, WORK_DIR, device, seed=SEED + FOLD,
                    pretrained=not DRY, deadline=deadline, resume_dirs=RESUME_DIRS, log=log)
    STATUS[name] = r
    if r["status"] != "done":
        break
ALL_DONE = all((WORK_DIR / "ckpt" / f"abl_{v}_fold{FOLD}.done.json").exists() for v in VARIANTS)
log(f"status: { {k: v['status'] for k, v in STATUS.items()} } | all variants trained: {ALL_DONE}")

In [ ]:
# ----------------------------------------------------------------------------------------------- evaluation
if not ALL_DONE:
    print("Not every variant is trained yet: save this version, add its output as input and run again.")
else:
    groups = {s: i % 4 for i, s in enumerate(VA)}                 # scorer cross-fitting inside fold 0
    R, rows = {}, []
    for name in VARIANTS:
        c = variant_cfg(name)
        info = json.loads((WORK_DIR / "ckpt" / f"abl_{name}_fold{FOLD}.done.json").read_text())
        prob = pl.oof_probabilities(c, VA, {s: FOLD for s in VA}, {FOLD: WORK_DIR / "ckpt" / f"abl_{name}_fold{FOLD}.pt"},
                                    CACHE_DIR, WORK_DIR / f"prob_{name}", device, log=log)
        C = pl.candidates(prob, VA, ann, CACHE_DIR, c, workers=WORKERS, log=log)
        R[name] = pl.evaluate_scorer(C, VA, groups, c, SEED)
        s_all, s_sc = ev.summary(R[name]["P_all"]), ev.summary(R[name]["P"])
        rows.append(dict(variant=name, best_val_iou=round(info["best"], 4), pq_all=round(s_all["PQ"], 4),
                         pq=round(s_sc["PQ"], 4), sq=round(s_sc["SQ"], 3), rq=round(s_sc["RQ"], 3),
                         tp=s_sc["TP"], fp=s_sc["FP"], fn=s_sc["FN"], tau=R[name]["tau"],
                         encoder_fp32=info.get("encoder_fp32", False)))
    T = pd.DataFrame(rows)
    comp = []
    for name in VARIANTS:
        if name == "ref":
            continue
        b, ba = ev.paired_bootstrap(R[name]["P"], R["ref"]["P"]), ev.paired_bootstrap(R[name]["P_all"], R["ref"]["P_all"])
        comp.append(dict(comparison=f"{name} - ref", d_pq=round(b["diff"], 4), ci95=f"[{b['lo']:+.4f}, {b['hi']:+.4f}]",
                         d_pq_all=round(ba["diff"], 4), ci95_all=f"[{ba['lo']:+.4f}, {ba['hi']:+.4f}]",
                         adopt=bool(b["lo"] > 0 and b["diff"] > 0.003)))
    K = pd.DataFrame(comp)
    log("\n" + T.to_string(index=False) + "\n\n" + K.to_string(index=False))
    T.to_csv(WORK_DIR / "ablation_results.csv", index=False); K.to_csv(WORK_DIR / "ablation_comparisons.csv", index=False)
    with open(WORK_DIR / "ablation_parts.pkl", "wb") as fh:
        pickle.dump({n: dict(P=r["P"], P_all=r["P_all"], tau=r["tau"]) for n, r in R.items()} | {"stems": VA}, fh)
    hist = []
    for name in VARIANTS:
        info = json.loads((WORK_DIR / "ckpt" / f"abl_{name}_fold{FOLD}.done.json").read_text())
        hist += [dict(variant=name, **h) for h in info["history"]]
    pd.DataFrame(hist).to_csv(WORK_DIR / "ablation_history.csv", index=False)